In [ ]:
# טעינת Dataset של CodeSearchNet עבור שפת Python
from datasets import load_dataset

raw_datasets = load_dataset("code_search_net", "python")


# הצגת מבנה ה-train split כדי לראות אילו עמודות יש בדאטה
raw_datasets["train"]


# הדפסת דוגמה אחת של פונקציה מלאה מתוך הדאטה
print(raw_datasets["train"][123456]["whole_func_string"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


def oauth_token_create(self, data, **kwargs):
        "https://developer.zendesk.com/rest_api/docs/core/oauth_tokens#create-token"
        api_path = "/api/v2/oauth/tokens.json"
        return self.call(api_path, method="POST", data=data, **kwargs)


In [ ]:


# דרך לא מומלצת: יצירת רשימת batches אבל טעינת כל הדאטה לזיכרון
# אל תריצי את זה אם הדאטה גדול
# training_corpus = [
#     raw_datasets["train"][i: i + 1000]["whole_func_string"]
#     for i in range(0, len(raw_datasets["train"]), 1000)
# ]


# יצירת generator expression כדי לטעון כל פעם 1000 טקסטים בלבד
training_corpus = (
    raw_datasets["train"][i: i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

In [ ]:
# דוגמה שמראה ש-generator אפשר לקרוא רק פעם אחת
gen = (i for i in range(10))

print(list(gen))
print(list(gen))


# פתרון טוב יותר: פונקציה שמחזירה generator חדש בכל קריאה
def get_training_corpus():
  """
  This function yields batches of data.
  """
    return (
        raw_datasets["train"][i: i + 1000]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), 1000)
    )


training_corpus = get_training_corpus()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


In [ ]:
# אותה לוגיקה, אבל עם yield — יותר גמיש ונוח ללוגיקה מורכבת
def get_training_corpus():
    dataset = raw_datasets["train"]

    for start_idx in range(0, len(dataset), 1000):
        samples = dataset[start_idx: start_idx + 1000]
        yield samples["whole_func_string"]


# יצירת הקורפוס לאימון מתוך פונקציית ה-yield
training_corpus = get_training_corpus()

In [ ]:
# טעינת tokenizer קיים של GPT-2 כדי לא להתחיל מאפס
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")


# יצירת דוגמת קוד Python כדי לבדוק איך tokenizer הישן מפרק אותה
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''


# בדיקה איך ה-tokenizer הישן מפרק את דוגמת הקוד לטוקנים
tokens = old_tokenizer.tokenize(example)

print(tokens)

['def', 'Ġadd', '_', 'n', 'umbers', '(', 'a', ',', 'Ġb', '):', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġ"""', 'Add', 'Ġthe', 'Ġtwo', 'Ġnumbers', 'Ġ`', 'a', '`', 'Ġand', 'Ġ`', 'b', '`', '."', '""', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']


In [ ]:
# אימון tokenizer חדש על הקורפוס שלנו, עם vocabulary בגודל 52,000
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)


# בדיקה איך ה-tokenizer החדש מפרק את אותה דוגמה
tokens = tokenizer.tokenize(example)

print(tokens)

['def', 'Ġadd', '_', 'numbers', '(', 'a', ',', 'Ġb', '):', 'ĊĠĠĠ', 'Ġ"""', 'Add', 'Ġthe', 'Ġtwo', 'Ġnumbers', 'Ġ`', 'a', '`', 'Ġand', 'Ġ`', 'b', '`."""', 'ĊĠĠĠ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']


In [ ]:
# השוואת מספר הטוקנים בין tokenizer חדש לבין tokenizer ישן
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))


# יצירת דוגמת קוד מורכבת יותר כדי לבדוק tokenization על class, init, self, indentation וכו'
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """


# בדיקה איך ה-tokenizer החדש מפרק את הדוגמה המורכבת
tokenizer.tokenize(example)


# שמירת ה-tokenizer החדש לתיקייה מקומית
tokenizer.save_pretrained("code-search-net-tokenizer")


27
36


('code-search-net-tokenizer/tokenizer_config.json',
 'code-search-net-tokenizer/tokenizer.json')

In [ ]:
# # התחברות ל-HuggingFace Hub מתוך notebook
# from huggingface_hub import notebook_login

# notebook_login()


# # אם לא עובדים מתוך notebook, מתחברים דרך הטרמינל עם הפקודה הזו
# # huggingface-cli login


# # העלאת ה-tokenizer החדש ל-HuggingFace Hub
# tokenizer.push_to_hub("code-search-net-tokenizer")


# # טעינת ה-tokenizer מה-Hub אחרי שהעלינו אותו
# # צריך להחליף את huggingface-course בשם המשתמש שלך ב-HuggingFace
# tokenizer = AutoTokenizer.from_pretrained(
#     "huggingface-course/code-search-net-tokenizer"
# )

פרק 6.3

In [ ]:
# טעינת tokenizer מהיר של BERT כדי לעבוד עם offset mapping ויכולות מתקדמות
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


# טקסט לדוגמה שנרצה לנתח
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."


# ביצוע tokenization רגיל (מחזיר BatchEncoding)
encoding = tokenizer(example)


# בדיקה איזה סוג אובייקט חזר
print(type(encoding))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [ ]:
# בדיקה אם ה-tokenizer הוא fast tokenizer
print(tokenizer.is_fast)
print(encoding.is_fast)


# קבלת רשימת הטוקנים ישירות (בלי decode)
print(encoding.tokens())


# קבלת word_ids — איזה token שייך לאיזה מילה מקורית
print(encoding.word_ids())


# המרת word index לטווח תווים בטקסט המקורי
start, end = encoding.word_to_chars(3)
print(example[start:end])

True
True
['[CLS]', 'My', 'name', 'is', 'S', '##yl', '##va', '##in', 'and', 'I', 'work', 'at', 'Hu', '##gging', 'Face', 'in', 'Brooklyn', '.', '[SEP]']
[None, 0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]
Sylvain


In [ ]:
# טעינת pipeline ל-NER (זיהוי ישויות בשם)
from transformers import pipeline

token_classifier = pipeline("token-classification")


# הרצת NER pipeline על הטקסט
print(token_classifier(example))


# הרצת pipeline עם aggregation (קיבוץ טוקנים למילים)
token_classifier = pipeline("token-classification", aggregation_strategy="simple")
print(token_classifier(example))


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'entity': 'I-PER', 'score': np.float32(0.99938285), 'index': 4, 'word': 'S', 'start': 11, 'end': 12}, {'entity': 'I-PER', 'score': np.float32(0.99815494), 'index': 5, 'word': '##yl', 'start': 12, 'end': 14}, {'entity': 'I-PER', 'score': np.float32(0.99590707), 'index': 6, 'word': '##va', 'start': 14, 'end': 16}, {'entity': 'I-PER', 'score': np.float32(0.99923277), 'index': 7, 'word': '##in', 'start': 16, 'end': 18}, {'entity': 'I-ORG', 'score': np.float32(0.9738931), 'index': 12, 'word': 'Hu', 'start': 33, 'end': 35}, {'entity': 'I-ORG', 'score': np.float32(0.976115), 'index': 13, 'word': '##gging', 'start': 35, 'end': 40}, {'entity': 'I-ORG', 'score': np.float32(0.9887976), 'index': 14, 'word': 'Face', 'start': 41, 'end': 45}, {'entity': 'I-LOC', 'score': np.float32(0.9932106), 'index': 16, 'word': 'Brooklyn', 'start': 49, 'end': 57}]


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER', 'score': np.float32(0.9981694), 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': np.float32(0.9796019), 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': np.float32(0.9932106), 'word': 'Brooklyn', 'start': 49, 'end': 57}]


In [ ]:
# טעינת tokenizer + model בצורה ידנית (בלי pipeline)
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)


# tokenization של הקלט בפורמט שמתאים למודל (tensor)
inputs = tokenizer(example, return_tensors="pt")


# העברת הקלט דרך המודל
outputs = model(**inputs)


# בדיקת הצורות של הקלט והפלט
print(inputs["input_ids"].shape)
print(outputs.logits.shape)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 19])
torch.Size([1, 19, 9])


In [ ]:
# המרת logits להסתברויות + חיזוי התווית לכל token
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()

print(predictions)


# הצגת מיפוי בין אינדקסים לתוויות (labels)
print(model.config.id2label)


# שחזור תוצאות כמו pipeline (ללא offsets עדיין)
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        results.append({
            "entity": label,
            "score": probabilities[idx][pred],
            "word": tokens[idx]
        })

print(results)

[0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 0, 8, 0, 0]
{0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
[{'entity': 'I-PER', 'score': 0.9993828535079956, 'word': 'S'}, {'entity': 'I-PER', 'score': 0.9981548190116882, 'word': '##yl'}, {'entity': 'I-PER', 'score': 0.995907187461853, 'word': '##va'}, {'entity': 'I-PER', 'score': 0.9992327690124512, 'word': '##in'}, {'entity': 'I-ORG', 'score': 0.9738931059837341, 'word': 'Hu'}, {'entity': 'I-ORG', 'score': 0.9761149883270264, 'word': '##gging'}, {'entity': 'I-ORG', 'score': 0.9887974858283997, 'word': 'Face'}, {'entity': 'I-LOC', 'score': 0.99321049451828, 'word': 'Brooklyn'}]


In [ ]:
# קבלת offset mapping (מיפוי טוקן → מיקום בטקסט המקורי)
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)

print(inputs_with_offsets["offset_mapping"])


# דוגמה: חילוץ substring לפי offset
print(example[12:14])


# שחזור תוצאות כולל start/end (כמו pipeline)
results = []

tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        start, end = offsets[idx]
        results.append({
            "entity": label,
            "score": probabilities[idx][pred],
            "word": tokens[idx],
            "start": start,
            "end": end
        })

print(results)

[(0, 0), (0, 2), (3, 7), (8, 10), (11, 12), (12, 14), (14, 16), (16, 18), (19, 22), (23, 24), (25, 29), (30, 32), (33, 35), (35, 40), (41, 45), (46, 48), (49, 57), (57, 58), (0, 0)]
yl
[{'entity': 'I-PER', 'score': 0.9993828535079956, 'word': 'S', 'start': 11, 'end': 12}, {'entity': 'I-PER', 'score': 0.9981548190116882, 'word': '##yl', 'start': 12, 'end': 14}, {'entity': 'I-PER', 'score': 0.995907187461853, 'word': '##va', 'start': 14, 'end': 16}, {'entity': 'I-PER', 'score': 0.9992327690124512, 'word': '##in', 'start': 16, 'end': 18}, {'entity': 'I-ORG', 'score': 0.9738931059837341, 'word': 'Hu', 'start': 33, 'end': 35}, {'entity': 'I-ORG', 'score': 0.9761149883270264, 'word': '##gging', 'start': 35, 'end': 40}, {'entity': 'I-ORG', 'score': 0.9887974858283997, 'word': 'Face', 'start': 41, 'end': 45}, {'entity': 'I-LOC', 'score': 0.99321049451828, 'word': 'Brooklyn', 'start': 49, 'end': 57}]


In [ ]:
# קיבוץ ישויות (Sylvain, Hugging Face וכו') לפי offset mapping
import numpy as np

results = []

idx = 0
while idx < len(predictions):
    pred = predictions[idx]
    label = model.config.id2label[pred]

    if label != "O":
        label = label[2:]  # מוריד B- / I-
        start, _ = offsets[idx]

        all_scores = []

        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])
            _, end = offsets[idx]
            idx += 1

        score = np.mean(all_scores).item()
        word = example[start:end]

        results.append({
            "entity_group": label,
            "score": score,
            "word": word,
            "start": start,
            "end": end
        })

    idx += 1

print(results)

[{'entity_group': 'PER', 'score': 0.998169407248497, 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': 0.9796018600463867, 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': 0.99321049451828, 'word': 'Brooklyn', 'start': 49, 'end': 57}]


פרק 6.4

In [ ]:
# שימוש ב־pipeline מוכן לשאלה-תשובה (QA)
from transformers import pipeline

question_answerer = pipeline("question-answering")

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch, and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""

question = "Which deep learning libraries back 🤗 Transformers?"

print(question_answerer(question=question, context=context))


# שימוש ב־pipeline על context ארוך מאוד (הוא יודע לפצל לבד)
long_context = """... (long text) ..."""
print(question_answerer(question=question, context=long_context))


# טעינת tokenizer ומודל QA ידנית
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_checkpoint = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)


# tokenization של question + context יחד
inputs = tokenizer(question, context, return_tensors="pt")


# העברת הקלט דרך המודל
outputs = model(**inputs)


# קבלת logits של התחלה וסוף
start_logits = outputs.start_logits
end_logits = outputs.end_logits

print(start_logits.shape, end_logits.shape)


# יצירת mask כדי להתעלם מטוקנים שלא שייכים ל-context
import torch

sequence_ids = inputs.sequence_ids()

mask = [i != 1 for i in sequence_ids]
mask[0] = False
mask = torch.tensor(mask)[None]

start_logits[mask] = -10000
end_logits[mask] = -10000


# המרת logits להסתברויות
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)[0]
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)[0]


# חישוב כל הזוגות האפשריים (start * end)
scores = start_probabilities[:, None] * end_probabilities[None, :]


# ביטול מקרים שבהם start > end
scores = torch.triu(scores)


# מציאת הזוג הכי טוב
max_index = scores.argmax().item()

start_index = max_index // scores.shape[1]
end_index = max_index % scores.shape[1]

print(scores[start_index, end_index])


# שימוש ב-offset mapping כדי לחזור לטקסט המקורי
inputs_with_offsets = tokenizer(question, context, return_offsets_mapping=True)

offsets = inputs_with_offsets["offset_mapping"]

start_char, _ = offsets[start_index]
_, end_char = offsets[end_index]

answer = context[start_char:end_char]


# יצירת התוצאה הסופית
result = {
    "answer": answer,
    "start": start_char,
    "end": end_char,
    "score": scores[start_index, end_index],
}

print(result)


# בדיקה כמה טוקנים יש ב-context ארוך (לפני חיתוך)
inputs = tokenizer(question, long_context)
print(len(inputs["input_ids"]))


# חיתוך context ארוך (רק ה-context, לא השאלה)
inputs = tokenizer(
    question,
    long_context,
    max_length=384,
    truncation="only_second"
)

print(tokenizer.decode(inputs["input_ids"]))


# דוגמה לפיצול טקסט ארוך עם overlap (stride)
sentence = "This sentence is not too long but we are going to split it anyway."

inputs = tokenizer(
    sentence,
    truncation=True,
    return_overflowing_tokens=True,
    max_length=6,
    stride=2
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))


# בדיקה של overflow mapping (איזה chunk שייך לאיזה משפט)
print(inputs.keys())
print(inputs["overflow_to_sample_mapping"])


# דוגמה עם כמה משפטים
sentences = [
    "This sentence is not too long but we are going to split it anyway.",
    "This sentence is shorter but will still get split.",
]

inputs = tokenizer(
    sentences,
    truncation=True,
    return_overflowing_tokens=True,
    max_length=6,
    stride=2
)

print(inputs["overflow_to_sample_mapping"])


# tokenization מלא ל-QA עם context ארוך + offsets + overlap
inputs = tokenizer(
    question,
    long_context,
    stride=128,
    max_length=384,
    padding="longest",
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)


# הוצאת מידע שלא צריך למודל
_ = inputs.pop("overflow_to_sample_mapping")
offsets = inputs.pop("offset_mapping")


# המרה ל-tensors
inputs = inputs.convert_to_tensors("pt")

print(inputs["input_ids"].shape)


# העברה דרך המודל (עכשיו יש כמה chunks)
outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits

print(start_logits.shape, end_logits.shape)


# masking כולל padding
sequence_ids = inputs.sequence_ids()

mask = [i != 1 for i in sequence_ids]
mask[0] = False

mask = torch.logical_or(
    torch.tensor(mask)[None],
    (inputs["attention_mask"] == 0)
)

start_logits[mask] = -10000
end_logits[mask] = -10000


# softmax לכל chunk
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)


# מציאת התשובה הכי טובה בכל chunk
candidates = []

for start_probs, end_probs in zip(start_probabilities, end_probabilities):
    scores = start_probs[:, None] * end_probs[None, :]
    idx = torch.triu(scores).argmax().item()

    start_idx = idx // scores.shape[1]
    end_idx = idx % scores.shape[1]
    score = scores[start_idx, end_idx].item()

    candidates.append((start_idx, end_idx, score))

print(candidates)


# המרה מ-token spans ל-character spans בטקסט המקורי
for candidate, offset in zip(candidates, offsets):
    start_token, end_token, score = candidate

    start_char, _ = offset[start_token]
    _, end_char = offset[end_token]

    answer = long_context[start_char:end_char]

    result = {
        "answer": answer,
        "start": start_char,
        "end": end_char,
        "score": score,
    }

    print(result)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

torch.Size([1, 67]) torch.Size([1, 67])
tensor(0.9803, grad_fn=<SelectBackward0>)
{'answer': 'Jax, PyTorch, and TensorFlow', 'start': 78, 'end': 106, 'score': tensor(0.9803, grad_fn=<SelectBackward0>)}
21
[CLS] Which deep learning libraries back [UNK] Transformers? [SEP]... ( long text )... [SEP]
[CLS] This sentence is not [SEP]
[CLS] is not too long [SEP]
[CLS] too long but we [SEP]
[CLS] but we are going [SEP]
[CLS] are going to split [SEP]
[CLS] to split it anyway [SEP]
[CLS] it anyway. [SEP]
KeysView({'input_ids': [[101, 1188, 5650, 1110, 1136, 102], [101, 1110, 1136, 1315, 1263, 102], [101, 1315, 1263, 1133, 1195, 102], [101, 1133, 1195, 1132, 1280, 102], [101, 1132, 1280, 1106, 3325, 102], [101, 1106, 3325, 1122, 4050, 102], [101, 1122, 4050, 119, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 

6.5

In [ ]:
# טעינת tokenizer (uncased → עושה lowercase + הסרת accents)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


# בדיקה איזה אובייקט עומד מאחורי ה-tokenizer (מהספרייה של tokenizers)
print(type(tokenizer.backend_tokenizer))


# בדיקה איך normalization עובד בפועל (lowercase + הסרת accents)
print(tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))


# טעינת tokenizer עם שמירה על אותיות גדולות (cased)
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


# בדיקה איך normalization שונה ב-cased tokenizer
print(tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))


# בדיקה של pre-tokenization ב-BERT (פיצול לפי רווחים וסימני פיסוק)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?"))


# בדיקה של pre-tokenization ב-GPT-2 (שומר רווחים עם סימן Ġ)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?"))


# בדיקה של pre-tokenization ב-T5 (SentencePiece)
tokenizer = AutoTokenizer.from_pretrained("t5-small")

print(tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?"))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<class 'tokenizers.Tokenizer'>
hello how are u?
Héllò hôw are ü?
[('Hello', (0, 5)), (',', (5, 6)), ('how', (7, 10)), ('are', (11, 14)), ('you', (16, 19)), ('?', (19, 20))]
[('Hello', (0, 5)), (',', (5, 6)), ('Ġhow', (6, 10)), ('Ġare', (10, 14)), ('Ġ', (14, 15)), ('Ġyou', (15, 19)), ('?', (19, 20))]


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[('▁Hello,', (0, 6)), ('▁how', (7, 10)), ('▁are', (11, 14)), ('▁you?', (16, 20))]


פרק 6.6

In [ ]:
# יצירת קורפוס קטן לדוגמה לאימון BPE
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]


# טעינת tokenizer של GPT-2 לצורך pre-tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")


# חישוב תדירות של כל "מילה" אחרי pre-tokenization
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)


# בניית האלפבית (כל התווים שמופיעים בקורפוס)
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)

alphabet.sort()

print(alphabet)


# יצירת vocabulary ראשוני (כולל token מיוחד)
vocab = ["<|endoftext|>"] + alphabet.copy()


# פיצול כל מילה לרשימת תווים (שלב התחלה של BPE)
splits = {word: [c for c in word] for word in word_freqs.keys()}


# פונקציה לחישוב תדירות של זוגות תווים (pairs)
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]

        if len(split) == 1:
            continue

        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq

    return pair_freqs


# חישוב זוגות תווים ראשוניים
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break


# מציאת הזוג הכי נפוץ
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)


# הוספת merge ראשון ל-vocab
merges = {(best_pair[0], best_pair[1]): best_pair[0] + best_pair[1]}
vocab.append(best_pair[0] + best_pair[1])


# פונקציה שממזגת זוג תווים בכל הקורפוס
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]

        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2:]
            else:
                i += 1

        splits[word] = split

    return splits


# ביצוע merge ראשון בפועל
splits = merge_pair(best_pair[0], best_pair[1], splits)

print(splits[list(splits.keys())[0]])


# המשך אימון עד שמגיעים לגודל vocab רצוי
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)

    best_pair = ""
    max_freq = None

    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq

    splits = merge_pair(best_pair[0], best_pair[1], splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])


# הדפסת כל ה-merge rules שנלמדו
print(merges)


# הדפסת ה-vocabulary הסופי
print(vocab)


# פונקציה ל-tokenization לפי merge rules שנלמדו
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]

    splits = [[l for l in word] for word in pre_tokenized_text]

    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2:]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])


# בדיקה על טקסט חדש
print(tokenize("This is not a token."))

defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})
[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']
('T', 'h'): 3
('h', 'i'): 3
('i', 's'): 5
('Ġ', 'i'): 2
('Ġ', 't'): 7
('t', 'h'): 3
('Ġ', 't') 7
['T', 'h', 'i', 's']
{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): '

6.7

In [ ]:
# יצירת קורפוס קטן לדוגמה לאימון WordPiece
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]


# טעינת tokenizer של BERT לצורך pre-tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


# חישוב תדירות של כל "מילה" אחרי pre-tokenization
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)


# בניית האלפבית (אות ראשונה רגילה, שאר האותיות עם ##)
alphabet = []

for word in word_freqs.keys():
    if word[0] not in alphabet:
        alphabet.append(word[0])
    for letter in word[1:]:
        if f"##{letter}" not in alphabet:
            alphabet.append(f"##{letter}")

alphabet.sort()
print(alphabet)


# יצירת vocabulary ראשוני (כולל tokens מיוחדים של BERT)
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()


# פיצול כל מילה לתווים עם prefix ## למעט הראשון
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}


# פונקציה לחישוב score של זוגות (הבדל מרכזי מ-BPE!)
def compute_pair_scores(splits):
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]

        if len(split) == 1:
            letter_freqs[split[0]] += freq
            continue

        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            letter_freqs[split[i]] += freq
            pair_freqs[pair] += freq

        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }

    return scores


# חישוב score ראשוני של זוגות
pair_scores = compute_pair_scores(splits)

for i, key in enumerate(pair_scores.keys()):
    print(f"{key}: {pair_scores[key]}")
    if i >= 5:
        break


# מציאת הזוג עם הציון הכי גבוה
best_pair = ""
max_score = None

for pair, score in pair_scores.items():
    if max_score is None or max_score < score:
        best_pair = pair
        max_score = score

print(best_pair, max_score)


# הוספת merge ראשון ל-vocab
new_token = best_pair[0] + best_pair[1][2:] if best_pair[1].startswith("##") else best_pair[0] + best_pair[1]
vocab.append(new_token)


# פונקציה שממזגת זוג תווים
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]

        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] + split[i + 2:]
            else:
                i += 1

        splits[word] = split

    return splits


# ביצוע merge ראשון בפועל
splits = merge_pair(best_pair[0], best_pair[1], splits)
print(splits[list(splits.keys())[0]])


# המשך אימון עד גודל vocab רצוי
vocab_size = 70

while len(vocab) < vocab_size:
    scores = compute_pair_scores(splits)

    best_pair = ""
    max_score = None

    for pair, score in scores.items():
        if max_score is None or max_score < score:
            best_pair = pair
            max_score = score

    splits = merge_pair(best_pair[0], best_pair[1], splits)

    new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )

    vocab.append(new_token)


# הדפסת ה-vocab שנלמד
print(vocab)


# פונקציה ל-tokenization של מילה אחת (Longest match!)
def encode_word(word):
    tokens = []

    while len(word) > 0:
        i = len(word)

        while i > 0 and word[:i] not in vocab:
            i -= 1

        if i == 0:
            return ["[UNK]"]

        tokens.append(word[:i])
        word = word[i:]

        if len(word) > 0:
            word = f"##{word}"

    return tokens


# בדיקה על מילים
print(encode_word("Hugging"))
print(encode_word("HOgging"))


# פונקציה ל-tokenization של טקסט מלא
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]

    encoded_words = [encode_word(word) for word in pre_tokenized_text]

    return sum(encoded_words, [])


# בדיקה על משפט
print(tokenize("This is the Hugging Face course!"))

defaultdict(<class 'int'>, {'This': 3, 'is': 2, 'the': 1, 'Hugging': 1, 'Face': 1, 'Course': 1, '.': 4, 'chapter': 1, 'about': 1, 'tokenization': 1, 'section': 1, 'shows': 1, 'several': 1, 'tokenizer': 1, 'algorithms': 1, 'Hopefully': 1, ',': 1, 'you': 1, 'will': 1, 'be': 1, 'able': 1, 'to': 1, 'understand': 1, 'how': 1, 'they': 1, 'are': 1, 'trained': 1, 'and': 1, 'generate': 1, 'tokens': 1})
['##a', '##b', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##v', '##w', '##y', '##z', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'g', 'h', 'i', 's', 't', 'u', 'w', 'y']
('T', '##h'): 0.125
('##h', '##i'): 0.03409090909090909
('##i', '##s'): 0.02727272727272727
('i', '##s'): 0.1
('t', '##h'): 0.03571428571428571
('##h', '##e'): 0.011904761904761904
('a', '##b') 0.2
['T', '##h', '##i', '##s']
['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##a', '##b', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', 

פרק 6.8

In [ ]:
# יצירת קורפוס לדוגמה לאימון Unigram
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]


# טעינת tokenizer (SentencePiece-style דרך XLNet)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")


# חישוב תדירות מילים (אחרי pre-tokenization)
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)


# חישוב תדירות תווים ותתי-מילים (subwords)
char_freqs = defaultdict(int)
subwords_freqs = defaultdict(int)

for word, freq in word_freqs.items():
    for i in range(len(word)):
        char_freqs[word[i]] += freq
        for j in range(i + 2, len(word) + 1):
            subwords_freqs[word[i:j]] += freq


# מיון תתי-מילים לפי תדירות
sorted_subwords = sorted(subwords_freqs.items(), key=lambda x: x[1], reverse=True)
print(sorted_subwords[:10])


# יצירת vocabulary ראשוני גדול
token_freqs = list(char_freqs.items()) + sorted_subwords[: 300 - len(char_freqs)]
token_freqs = {token: freq for token, freq in token_freqs}


# חישוב הסתברויות (לוג) לכל token
from math import log

total_sum = sum([freq for token, freq in token_freqs.items()])
model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}


# פונקציה ל-tokenization של מילה באמצעות Viterbi (בחירת segmentation הכי טוב)
def encode_word(word, model):
    best_segmentations = [{"start": 0, "score": 1}] + [
        {"start": None, "score": None} for _ in range(len(word))
    ]

    for start_idx in range(len(word)):
        best_score_at_start = best_segmentations[start_idx]["score"]

        for end_idx in range(start_idx + 1, len(word) + 1):
            token = word[start_idx:end_idx]

            if token in model and best_score_at_start is not None:
                score = model[token] + best_score_at_start

                if (
                    best_segmentations[end_idx]["score"] is None
                    or best_segmentations[end_idx]["score"] > score
                ):
                    best_segmentations[end_idx] = {"start": start_idx, "score": score}

    segmentation = best_segmentations[-1]

    if segmentation["score"] is None:
        return ["<unk>"], None

    score = segmentation["score"]
    start = segmentation["start"]
    end = len(word)

    tokens = []

    while start != 0:
        tokens.insert(0, word[start:end])
        next_start = best_segmentations[start]["start"]
        end = start
        start = next_start

    tokens.insert(0, word[start:end])

    return tokens, score


# בדיקה על מילים
print(encode_word("Hopefully", model))
print(encode_word("This", model))


# חישוב loss של המודל על הקורפוס
def compute_loss(model):
    loss = 0
    for word, freq in word_freqs.items():
        _, word_loss = encode_word(word, model)
        loss += freq * word_loss
    return loss


print(compute_loss(model))


# חישוב השפעת הסרה של כל token על ה-loss
import copy

def compute_scores(model):
    scores = {}
    model_loss = compute_loss(model)

    for token, score in model.items():
        if len(token) == 1:
            continue

        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)

        scores[token] = compute_loss(model_without_token) - model_loss

    return scores


# בדיקה של השפעת tokens
scores = compute_scores(model)
print(scores["ll"])
print(scores["his"])


# הסרת tokens עם השפעה נמוכה עד שמגיעים לגודל vocab רצוי
percent_to_remove = 0.1

while len(model) > 100:
    scores = compute_scores(model)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1])

    for i in range(int(len(model) * percent_to_remove)):
        _ = token_freqs.pop(sorted_scores[i][0])

    total_sum = sum([freq for token, freq in token_freqs.items()])
    model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}


# פונקציה ל-tokenization של טקסט מלא
def tokenize(text, model):
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in words_with_offsets]

    encoded_words = [encode_word(word, model)[0] for word in pre_tokenized_text]

    return sum(encoded_words, [])


# בדיקה על משפט
print(tokenize("This is the Hugging Face course.", model))

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

defaultdict(<class 'int'>, {'▁This': 3, '▁is': 2, '▁the': 1, '▁Hugging': 1, '▁Face': 1, '▁Course.': 1, '▁chapter': 1, '▁about': 1, '▁tokenization.': 1, '▁section': 1, '▁shows': 1, '▁several': 1, '▁tokenizer': 1, '▁algorithms.': 1, '▁Hopefully,': 1, '▁you': 1, '▁will': 1, '▁be': 1, '▁able': 1, '▁to': 1, '▁understand': 1, '▁how': 1, '▁they': 1, '▁are': 1, '▁trained': 1, '▁and': 1, '▁generate': 1, '▁tokens.': 1})
[('▁t', 7), ('is', 5), ('er', 5), ('▁a', 5), ('▁to', 4), ('to', 4), ('en', 4), ('▁T', 3), ('▁Th', 3), ('▁Thi', 3)]
(['H', 'o', 'p', 'e', 'f', 'u', 'll', 'y'], 41.5157494601402)
(['This'], 6.288267030694535)
413.10377642940875
6.376412403623874
0.0
['▁This', '▁is', '▁the', '▁Hugging', '▁Face', '▁', 'c', 'ou', 'r', 's', 'e', '.']


פרק 6.8

In [ ]:
# טעינת WikiText-2 והכנת generator שמחזיר batches של 1000 טקסטים
from datasets import load_dataset

dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")


def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i : i + 1000]["text"]


# שמירת כל הטקסטים לקובץ טקסט מקומי
with open("wikitext-2.txt", "w", encoding="utf-8") as f:
    for i in range(len(dataset)):
        f.write(dataset[i]["text"] + "\n")


# ייבוא כל אבני הבניין של ספריית tokenizers ויצירת WordPiece tokenizer
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))


# הגדרת normalizer מוכן של BERT
tokenizer.normalizer = normalizers.BertNormalizer(lowercase=True)


# בניית normalizer ידנית: Unicode normalization + lowercase + הסרת accents
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents()]
)


# בדיקת פעולת הנרמול
print(tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))


# הגדרת pre-tokenizer מוכן של BERT
tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()


# הגדרת pre-tokenizer שמפצל לפי רווחים ופיסוק
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()


# בדיקת pre-tokenization
print(tokenizer.pre_tokenizer.pre_tokenize_str("Let's test my pre-tokenizer."))


# pre-tokenizer שמפצל רק לפי רווחים
pre_tokenizer = pre_tokenizers.WhitespaceSplit()
print(pre_tokenizer.pre_tokenize_str("Let's test my pre-tokenizer."))


# שילוב כמה pre-tokenizers ברצף
pre_tokenizer = pre_tokenizers.Sequence(
    [pre_tokenizers.WhitespaceSplit(), pre_tokenizers.Punctuation()]
)
print(pre_tokenizer.pre_tokenize_str("Let's test my pre-tokenizer."))


# הגדרת special tokens ו־trainer עבור WordPiece
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.WordPieceTrainer(vocab_size=25000, special_tokens=special_tokens)


# אימון tokenizer מתוך generator
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)


# אימון tokenizer מתוך קובץ טקסט
tokenizer.model = models.WordPiece(unk_token="[UNK]")
tokenizer.train(["wikitext-2.txt"], trainer=trainer)


# בדיקת tokenizer על משפט
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)


# שליפת IDs של special tokens
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print(cls_token_id, sep_token_id)


# הגדרת post-processing בסגנון BERT
tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)


# בדיקה אחרי הוספת special tokens
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)


# בדיקת זוג משפטים כולל type_ids
encoding = tokenizer.encode("Let's test this tokenizer...", "on a pair of sentences.")
print(encoding.tokens)
print(encoding.type_ids)


# הוספת decoder של WordPiece
tokenizer.decoder = decoders.WordPiece(prefix="##")


# בדיקת decode חזרה לטקסט
tokenizer.decode(encoding.ids)


# שמירת tokenizer לקובץ JSON
tokenizer.save("tokenizer.json")


# טעינת tokenizer מקובץ JSON
new_tokenizer = Tokenizer.from_file("tokenizer.json")


# עטיפת tokenizer ב־PreTrainedTokenizerFast לשימוש עם Transformers
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    # tokenizer_file="tokenizer.json", # You can load from the tokenizer file, alternatively
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)


# עטיפה ישירה כ־BertTokenizerFast
from transformers import BertTokenizerFast

wrapped_tokenizer = BertTokenizerFast(tokenizer_object=tokenizer)


# יצירת BPE tokenizer בסגנון GPT-2
tokenizer = Tokenizer(models.BPE())


# הגדרת byte-level pre-tokenizer של GPT-2
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)


# בדיקת pre-tokenization ב־BPE
tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!")


# הגדרת trainer עבור BPE ואימון מתוך generator
trainer = trainers.BpeTrainer(vocab_size=25000, special_tokens=["<|endoftext|>"])
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)


# אימון BPE מתוך קובץ טקסט
tokenizer.model = models.BPE()
tokenizer.train(["wikitext-2.txt"], trainer=trainer)


# בדיקת tokenization ב־BPE
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)


# הוספת post-processing של ByteLevel
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)


# בדיקת offsets עבור טוקן שמתחיל ברווח
sentence = "Let's test this tokenizer."
encoding = tokenizer.encode(sentence)
start, end = encoding.offsets[4]
sentence[start:end]


# הוספת decoder של ByteLevel
tokenizer.decoder = decoders.ByteLevel()


# בדיקת decode ב־BPE
tokenizer.decode(encoding.ids)


# עטיפת BPE tokenizer ב־PreTrainedTokenizerFast
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
)


# עטיפה ישירה כ־GPT2TokenizerFast
from transformers import GPT2TokenizerFast

wrapped_tokenizer = GPT2TokenizerFast(tokenizer_object=tokenizer)


# יצירת Unigram tokenizer
tokenizer = Tokenizer(models.Unigram())


# הגדרת normalizer בסגנון XLNet / SentencePiece
from tokenizers import Regex

tokenizer.normalizer = normalizers.Sequence(
    [
        normalizers.Replace("``", '"'),
        normalizers.Replace("''", '"'),
        normalizers.NFKD(),
        normalizers.StripAccents(),
        normalizers.Replace(Regex(" {2,}"), " "),
    ]
)


# הגדרת Metaspace pre-tokenizer
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()


# בדיקת pre-tokenization עם Metaspace
tokenizer.pre_tokenizer.pre_tokenize_str("Let's test the pre-tokenizer!")


# הגדרת special tokens ו־trainer עבור Unigram
special_tokens = ["<cls>", "<sep>", "<unk>", "<pad>", "<mask>", "<s>", "</s>"]
trainer = trainers.UnigramTrainer(
    vocab_size=25000, special_tokens=special_tokens, unk_token="<unk>"
)
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)


# אימון Unigram מתוך קובץ טקסט
tokenizer.model = models.Unigram()
tokenizer.train(["wikitext-2.txt"], trainer=trainer)


# בדיקת tokenization ב־Unigram
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)


# שליפת IDs של special tokens עבור XLNet
cls_token_id = tokenizer.token_to_id("<cls>")
sep_token_id = tokenizer.token_to_id("<sep>")
print(cls_token_id, sep_token_id)


# הגדרת post-processing בסגנון XLNet
tokenizer.post_processor = processors.TemplateProcessing(
    single="$A:0 <sep>:0 <cls>:2",
    pair="$A:0 <sep>:0 $B:1 <sep>:1 <cls>:2",
    special_tokens=[("<sep>", sep_token_id), ("<cls>", cls_token_id)],
)


# בדיקת זוג משפטים ב־Unigram כולל type_ids
encoding = tokenizer.encode("Let's test this tokenizer...", "on a pair of sentences!")
print(encoding.tokens)
print(encoding.type_ids)


# הוספת decoder של Metaspace
tokenizer.decoder = decoders.Metaspace()


# עטיפת Unigram tokenizer ב־PreTrainedTokenizerFast
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<cls>",
    sep_token="<sep>",
    mask_token="<mask>",
    padding_side="left",
)


# עטיפה ישירה כ־XLNetTokenizerFast
from transformers import XLNetTokenizerFast

wrapped_tokenizer = XLNetTokenizerFast(tokenizer_object=tokenizer)

hello how are u?
[('Let', (0, 3)), ("'", (3, 4)), ('s', (4, 5)), ('test', (6, 10)), ('my', (11, 13)), ('pre', (14, 17)), ('-', (17, 18)), ('tokenizer', (18, 27)), ('.', (27, 28))]
[("Let's", (0, 5)), ('test', (6, 10)), ('my', (11, 13)), ('pre-tokenizer.', (14, 28))]
['let', "'", 's', 'test', 'this', 'tok', '##eni', '##zer', '.']
2 3
['[CLS]', 'let', "'", 's', 'test', 'this', 'tok', '##eni', '##zer', '.', '[SEP]']
['[CLS]', 'let', "'", 's', 'test', 'this', 'tok', '##eni', '##zer', '...', '[SEP]', 'on', 'a', 'pair', 'of', 'sentences', '.', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
['L', 'et', "'", 's', 'Ġtest', 'Ġthis', 'Ġto', 'ken', 'izer', '.']
['▁Let', "'", 's', '▁test', '▁this', '▁to', 'ken', 'izer', '.']
0 1
['▁Let', "'", 's', '▁test', '▁this', '▁to', 'ken', 'izer', '.', '.', '.', '<sep>', '▁', 'on', '▁', 'a', '▁pair', '▁of', '▁sentence', 's', '!', '<sep>', '<cls>']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]
